In [1]:
# Setup:

import pandas as pd
from pathlib import Path

NETID = "cdamron3"

validation = pd.read_csv(
    f"{NETID}_validation_log.csv",
    keep_default_na=False
)

resolutions = pd.read_csv(
    f"{NETID}_disagreement_resolution.csv",
    keep_default_na=False
)

print(f"Validation records: {len(validation)}")
print(f"Disagreement records: {len(resolutions)}")


Validation records: 50
Disagreement records: 10


In [3]:
%pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 40.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Statistics for Agreements:

from pathlib import Path
from sklearn.metrics import cohen_kappa_score

validation = pd.read_csv(
    f"{NETID}_validation_log.csv",
    keep_default_na=False
)

resolutions = pd.read_csv(
    f"{NETID}_disagreement_resolution.csv",
    keep_default_na=False
)

print(f"Validation records: {len(validation)}")
print(f"Disagreement records: {len(resolutions)}")

FIELDS = [
    "HasRecovered",
    "WhyRecovered",
    "ActivityPattern",
    "Currentstatus",
    "RecentThemes"
]

CATEGORICAL_FIELDS = {
    "HasRecovered",
    "ActivityPattern",
    "Currentstatus"
}

PRE_COL = "PreDiscussAgreementstatus(Yes/No)"
POST_COL = "PostDiscussAgreementstatus(Yes/No)"

stats_rows = []

for field in FIELDS:

    field_data = validation[
        validation["Field"] == field
    ].copy()

    field_resolutions = resolutions[
        resolutions["Field"] == field
    ].copy()

    total = len(field_data)

    num_agree_pre = (
        field_data[PRE_COL].str.strip().str.lower() == "yes"
    ).sum()

    percent_pre = round(
        num_agree_pre / total * 100, 1
    )

    num_disagreements = total - num_agree_pre
    
    num_resolved = (
        field_resolutions[POST_COL]
        .str.strip()
        .str.lower()
        .eq("yes")
        .sum()
    )

    num_agree_post = num_agree_pre + num_resolved
    percent_post = round(
        num_agree_post / total * 100, 1
    )

    if field in CATEGORICAL_FIELDS:

        author_pre = field_data["AuthorValue"].tolist()
        validator_pre = field_data["ValidatorValue"].tolist()

        kappa_pre = cohen_kappa_score(
            author_pre,
            validator_pre
        )

        # Final agreed labels after adjudication
        author_post = []
        validator_post = []

        for _, row in field_data.iterrows():

            if row[PRE_COL].strip().lower() == "yes":
                author_post.append(row["AuthorValue"])
                validator_post.append(row["ValidatorValue"])

            else:
                adjudicated = row["AdjudicatedValue"]

                if not adjudicated:
                    raise ValueError(
                        f"Missing adjudication for {row['WOCProjectID']}, "
                        f"{field}"
                    )

                author_post.append(adjudicated)
                validator_post.append(adjudicated)

        kappa_post = cohen_kappa_score(
            author_post,
            validator_post
        )

        kappa_pre = (
            round(kappa_pre, 3)
            if pd.notna(kappa_pre)
            else "NA"
        )

        kappa_post = (
            round(kappa_post, 3)
            if pd.notna(kappa_post)
            else "NA"
        )

    else:
        kappa_pre = "NA"
        kappa_post = "NA"

    mean_resolution_time = "NA"

    notes = {
        "HasRecovered": (
            "Recovery classifications were consistent before discussion."
        ),
        "WhyRecovered": (
            "One explanation was refined after reviewing "
            "CI, documentation, and maintenance activity."
        ),
        "ActivityPattern": (
            "Trend interpretation caused several disagreements. "
            "One was resolved through an Irregular compromise."
        ),
        "Currentstatus": (
            "All classifications agreed before discussion "
            "after correcting the validation log."
        ),
        "RecentThemes": (
            "Differences involved identifying dominant commit themes "
            "and using GitHub rather than WoC commit evidence."
        )
    }

    stats_rows.append({
        "Field": field,
        "PercentAgree_Pre": percent_pre,
        "PercentAgree_Post": percent_post,
        "CohenKappa_Pre": kappa_pre,
        "CohenKappa_Post": kappa_post,
        "NumDisagreements": num_disagreements,
        "NumResolved": int(num_resolved),
        "MeanResolutionTime_min": mean_resolution_time,
        "Notes": notes[field]
    })

agreement_stats = pd.DataFrame(stats_rows)

agreement_stats.to_csv(
    f"{NETID}_agreement_stats.csv",
    index=False
)

display(agreement_stats)

print(f"Created: {NETID}_agreement_stats.csv")



Validation records: 50
Disagreement records: 10


,Field,PercentAgree_Pre,PercentAgree_Post,CohenKappa_Pre,CohenKappa_Post,NumDisagreements,NumResolved,MeanResolutionTime_min,Notes
0,HasRecovered,100.0,100.0,0.714,0.714,0,0,NA,Recovery classifications were consistent befor...
1,WhyRecovered,90.0,100.0,NA,NA,1,1,NA,One explanation was refined after reviewing CI...
2,ActivityPattern,50.0,100.0,0.0,0.444,5,5,NA,Trend interpretation caused several disagreeme...
3,Currentstatus,100.0,100.0,1.0,1.0,0,0,NA,All classifications agreed before discussion a...
4,RecentThemes,60.0,100.0,NA,NA,4,4,NA,Differences involved identifying dominant comm...


Created: cdamron3_agreement_stats.csv


In [9]:
# Checks to make sure everything went smoothly:

assert len(agreement_stats) == 5

assert agreement_stats["NumDisagreements"].sum() == 10

assert agreement_stats["NumResolved"].sum() == 10

assert (
    agreement_stats["PercentAgree_Post"] == 100.0
).all()

print("PART 3 AGREEMENT STATISTICS COMPLETE")
print("=" * 45)

display(agreement_stats)


PART 3 AGREEMENT STATISTICS COMPLETE


,Field,PercentAgree_Pre,PercentAgree_Post,CohenKappa_Pre,CohenKappa_Post,NumDisagreements,NumResolved,MeanResolutionTime_min,Notes
0,HasRecovered,100.0,100.0,0.714,0.714,0,0,NA,Recovery classifications were consistent befor...
1,WhyRecovered,90.0,100.0,NA,NA,1,1,NA,One explanation was refined after reviewing CI...
2,ActivityPattern,50.0,100.0,0.0,0.444,5,5,NA,Trend interpretation caused several disagreeme...
3,Currentstatus,100.0,100.0,1.0,1.0,0,0,NA,All classifications agreed before discussion a...
4,RecentThemes,60.0,100.0,NA,NA,4,4,NA,Differences involved identifying dominant comm...
